# Lab 13 — Part 2: Search Functions and Retrieval Approaches

**Learning goals:** Reuse the retrieval pipeline in a function, explore different queries, and compare sparse and dense retrieval.

## Setup — Run this cell first

The cell below recreates the small document collection and TF-IDF vectors used in Part 1.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."
]

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents)

## 1. Make a simple search function

Now combine the steps so you can search the document collection using any query.

In [2]:
def search(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    for i in top_ids:
        print(f"Score: {scores[i]:.3f} | {documents[i]}")

search("How can we measure similarity between text vectors?")

Score: 0.686 | Cosine similarity measures how similar two text vectors are.
Score: 0.148 | Named entity recognition identifies names, places, and organizations in text.


## 2. Sparse vs. dense retrieval (conceptual comparison)

| Sparse retrieval | Dense retrieval |
|---|---|
| Represents text using word-based features, e.g., TF-IDF | Represents text using learned embeddings |
| Works well when important query words appear in documents | Can identify similar meanings even when different words are used |
| Simple and lightweight | Usually requires a pretrained embedding model |

**Important limitation:** TF-IDF mainly matches words, not meanings. A query about *"automobiles"* may not retrieve a document about *"cars"* unless the words also overlap in other ways.

In modern RAG, retrieval can be sparse, dense, or a combination of both.

## 3. Does a high-ranked result always mean a good match?

The search function returns the **highest-scoring** documents, even when all the scores are zero. A high *rank* does not always mean a high *similarity score*.

For example, our collection is about NLP. What happens when we search for an unrelated topic like `"volcano eruption"`?

In [3]:
search("volcano eruption", k=2)

# Compare it with a relevant query
print("--- Relevant query ---")
search("cosine similarity", k=2)

Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.
--- Relevant query ---
Score: 0.594 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.


## 4. Improve search: skip results with a score of zero

We can improve the function by printing only documents with a **positive similarity score**.

In [4]:
def search_filtered(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    found = False
    for i in top_ids:
        if scores[i] > 0:
            print(f"Score: {scores[i]:.3f} | {documents[i]}")
            found = True
    if not found:
        print("No matching documents found.")

search_filtered("volcano eruption")
print("---")
search_filtered("cosine similarity")

No matching documents found.
---
Score: 0.594 | Cosine similarity measures how similar two text vectors are.


## 5. Practice questions — completed

The practice cell below answers both questions. Named entity recognition ranks its dedicated note first. After adding the BERT sentence, rebuild TF-IDF before searching because the vectorizer's vocabulary and document matrix must include the new document.

In [5]:
print("Exercise 1 — named entity recognition")
search("named entity recognition", k=2)

print("\nExercise 2 — add BERT and rebuild the index")
bert_note = "BERT is a transformer model that uses bidirectional context."
if bert_note not in documents:
    documents.append(bert_note)
# fit_transform rebuilds the vocabulary and every document vector.
document_vectors = vectorizer.fit_transform(documents)
print("Documents indexed:", len(documents))
print("TF-IDF matrix shape:", document_vectors.shape)
search("transformer models", k=3)
print("Rebuilding was necessary because the new document can add vocabulary terms and needs its own row in the vector matrix.")

Exercise 1 — named entity recognition
Score: 0.625 | Named entity recognition identifies names, places, and organizations in text.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.

Exercise 2 — add BERT and rebuild the index
Documents indexed: 7
TF-IDF matrix shape: (7, 45)
Score: 0.408 | BERT is a transformer model that uses bidirectional context.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.
Rebuilding was necessary because the new document can add vocabulary terms and needs its own row in the vector matrix.


In [6]:
# Practice: run your own queries here
search("named entity recognition", k=2)

Score: 0.625 | Named entity recognition identifies names, places, and organizations in text.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.
